# Frozen COGNIX Kaggle CUDA validation only
User authorized GPU 0 isolation: two T4 allocated, one visible to PyTorch.
No full graph training, TEST, conformal, scientific aggregation or source patching.


In [ ]:
# COGNIX VALIDATION ONLY. Never run future full commands in this notebook.
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

# Change only this input path to the private compact bundle attached to this notebook.
BUNDLE_INPUT = Path('/kaggle/input/datasets/adityar4685/cognix-kaggle-validation-v1')
ROOT = Path('/kaggle/working/cognix_validation')
OUTPUT = Path('/kaggle/working/carla_kaggle_environment_lock_v1')
EXPECTED_ARCHIVE_HASH = '4a256f93a1de49a8d1d988122e379bcbe8008a59910df28045ff08beebcbab39'
EXPECTED_BUNDLE_SEAL = '61544ab57409112695582fbfe170a9d832a550ae35f11336eabde9da6a5cc1f1'

def digest(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

assert BUNDLE_INPUT.is_dir(), 'Attach only the private compact validation input and set its exact path'
assert not ROOT.exists() and not OUTPUT.exists(), 'Use clean output paths; never overwrite evidence'
archive = BUNDLE_INPUT / 'cognix_kaggle_validation_v1.zip'
if archive.is_file():
    assert digest(archive) == EXPECTED_ARCHIVE_HASH, 'Archive SHA-256 mismatch'
    ROOT.mkdir()
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            assert (ROOT / info.filename).resolve().is_relative_to(ROOT.resolve()), 'Unsafe archive member'
        z.extractall(ROOT)
else:
    # Kaggle may unpack a uploaded dataset archive. Verify its anchored seal first.
    assert (BUNDLE_INPUT / 'SHA256SUMS').is_file(), 'Expected compact bundle files missing'
    assert digest(BUNDLE_INPUT / 'SHA256SUMS') == EXPECTED_BUNDLE_SEAL, 'Input bundle seal mismatch'
    shutil.copytree(BUNDLE_INPUT, ROOT)
assert digest(ROOT / 'SHA256SUMS') == EXPECTED_BUNDLE_SEAL, 'Bundle seal mismatch'
for line in (ROOT / 'SHA256SUMS').read_text().splitlines():
    expected, relative = line.split('  ', 1)
    path = (ROOT / relative).resolve()
    assert path.is_relative_to(ROOT.resolve()) and digest(path) == expected, 'Bundle file mismatch: ' + relative
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # user authorized one visible T4 from T4 x2
command = [sys.executable, str(ROOT / 'reports/carla_kaggle_validation_bundle_v1/validate_kaggle.py'),
           '--output', str(OUTPUT)]
completed = subprocess.run(command, cwd=ROOT)
assert completed.returncode == 0, 'Validation failed; preserve failure evidence and stop without fallback'
readiness = json.loads((OUTPUT / 'execution_readiness.json').read_text())
assert readiness['ready'] is True and readiness['full_training_executed'] is False
print((OUTPUT / 'report.md').read_text())
shutil.make_archive('/kaggle/working/carla_kaggle_validation_evidence_v1', 'zip', OUTPUT)
print('Download carla_kaggle_validation_evidence_v1.zip and review. Do not run the full experiment.')
